In [ ]:
# Support running the complete pipeline on Colab or Kaggle
# If you're running this notebook on Colab or Kaggle, you can uncomment the following 
# lines to clone the repository and navigate into it. This will allow you to run the complete 
# pipeline without needing to set up the environment manually.
# If you're running this notebook locally, you can ignore these lines and make sure 
# to set up the environment as described in the README.

# git clone https://github.com/HaianCao/ImageCaptioningE2E.git
# cd ImageCaptioningE2E

In [15]:
# Cấu hình chạy nằm trong configs/config.yaml, configs/task1_config.yaml, configs/task2_config.yaml.
# Cell này chỉ là ghi chú; các biến thực thi sẽ được nạp từ config ở cell sau.

# Chỉnh config YAML thay vì sửa giá trị trực tiếp trong notebook.

## 1. Cài đặt Môi trường (Setup)

In [16]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install transformers timm datasets tqdm omegaconf scikit-learn pyyaml wandb
%pip install opencv-python pillow matplotlib seaborn

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
import sys
import torch

current_dir = os.path.abspath(os.getcwd())
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

to_remove = [k for k in sys.modules.keys() if k == 'src' or k.startswith('src.')]
for k in to_remove:
    del sys.modules[k]

print(f"Project Root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

Project Root: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
PyTorch version: 2.10.0+cpu
Using Device: cpu


In [18]:
from pathlib import Path
import json
import random
import torch

from src.utils.config_loader import load_task_configs, print_config

BASE_CONFIG = load_task_configs("configs/config.yaml")
TASK1_CONFIG = load_task_configs("configs/config.yaml", "configs/task1_config.yaml")
TASK2_CONFIG = load_task_configs("configs/config.yaml", "configs/task2_config.yaml")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / BASE_CONFIG.paths.data_dir
RAW_DIR = PROJECT_ROOT / BASE_CONFIG.paths.raw_dir

TRAINING_MODE = str(BASE_CONFIG.pipeline.training_mode)
DOWNLOAD_DATA = bool(BASE_CONFIG.pipeline.download_data)
STRICT_SAMPLE_MODE = bool(BASE_CONFIG.sampling.strict_mode)
SAMPLE_SIZE = int(BASE_CONFIG.sampling.sample_size)
SAMPLE_SPLIT_RATIOS = tuple(float(x) for x in BASE_CONFIG.sampling.split_ratios)
SAMPLE_SEED = int(BASE_CONFIG.sampling.seed)
IMAGE_DOWNLOAD_MODE = "none" if STRICT_SAMPLE_MODE else str(BASE_CONFIG.sampling.image_download_mode)
PRE_EXTRACT_FEATURES = bool(BASE_CONFIG.pipeline.pre_extract_features)

FEATURE_BATCH_SIZE = int(BASE_CONFIG.feature_extraction.batch_size)
FEATURE_RESIZE_SIZE = int(BASE_CONFIG.feature_extraction.resize_size)
FEATURE_CROP_SIZE = int(BASE_CONFIG.feature_extraction.crop_size)
FEATURE_MEAN = [float(x) for x in BASE_CONFIG.image.mean]
FEATURE_STD = [float(x) for x in BASE_CONFIG.image.std]
FULL_SPLIT_RATIOS = (float(BASE_CONFIG.split.train), float(BASE_CONFIG.split.val), float(BASE_CONFIG.split.test))
PREPROCESS_SPLIT_RATIOS = SAMPLE_SPLIT_RATIOS if STRICT_SAMPLE_MODE else FULL_SPLIT_RATIOS

MAX_SAMPLES = None if BASE_CONFIG.sampling.max_samples is None else int(BASE_CONFIG.sampling.max_samples)

if STRICT_SAMPLE_MODE:
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_sample"
else:
    PROCESSED_DIR = PROJECT_ROOT / BASE_CONFIG.paths.processed_dir

CHECKPOINT_DIR = PROJECT_ROOT / BASE_CONFIG.paths.checkpoint_dir
LOG_DIR = PROJECT_ROOT / BASE_CONFIG.paths.log_dir

TASK1_PROCESSED_DIR = PROCESSED_DIR / "task1"
TASK2_PROCESSED_DIR = PROCESSED_DIR / "task2"
TASK1_FEATURE_DIR = TASK1_PROCESSED_DIR / TASK1_CONFIG.dataset.feature_cache_dir
TASK2_FEATURE_DIR = TASK2_PROCESSED_DIR / TASK2_CONFIG.dataset.feature_cache_dir

TASK1_BATCH_SIZE = int(TASK1_CONFIG.training.batch_size)
TASK2_BATCH_SIZE = int(TASK2_CONFIG.training.batch_size)
TASK1_LR = float(TASK1_CONFIG.training.lr)
TASK2_LR = float(TASK2_CONFIG.training.lr)
TASK1_MAX_EPOCHS = int(TASK1_CONFIG.training.max_epochs)
TASK2_MAX_EPOCHS = int(TASK2_CONFIG.training.max_epochs)
TASK1_ROI_SIZE = int(BASE_CONFIG.image.roi_size)
TASK2_ROI_SIZE = int(BASE_CONFIG.image.roi_size)

TASK1_OBJECT_HIDDEN_DIM = int(TASK1_CONFIG.model.object_hidden_dim)
TASK1_OBJECT_DROPOUT = float(TASK1_CONFIG.model.object_dropout)
TASK1_OBJECT_NUM_LAYERS = int(TASK1_CONFIG.model.object_num_layers)
TASK1_ATTRIBUTE_HIDDEN_DIM = int(TASK1_CONFIG.model.attribute_hidden_dim)
TASK1_ATTRIBUTE_DROPOUT = float(TASK1_CONFIG.model.attribute_dropout)
TASK1_ATTRIBUTE_NUM_LAYERS = int(TASK1_CONFIG.model.attribute_num_layers)
TASK1_OBJECT_WEIGHT = float(TASK1_CONFIG.loss.object_weight)
TASK1_ATTRIBUTE_WEIGHT = float(TASK1_CONFIG.loss.attribute_weight)
TASK1_ATTRIBUTE_POS_WEIGHT = float(TASK1_CONFIG.loss.attribute_pos_weight)

TASK2_FUSION_METHOD = str(TASK2_CONFIG.model.fusion_method)
TASK2_HIDDEN_DIM = int(TASK2_CONFIG.model.hidden_dim)
TASK2_DROPOUT = float(TASK2_CONFIG.model.dropout)
TASK2_NUM_LAYERS = int(TASK2_CONFIG.model.num_layers)
TASK2_ATTENTION_HEADS = int(TASK2_CONFIG.model.attention_heads)

DEVICE = BASE_CONFIG.device if BASE_CONFIG.device == "cpu" or torch.cuda.is_available() else "cpu"
print_config(BASE_CONFIG)
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {DEVICE}")
print(f"Training mode: {TRAINING_MODE}")
print(f"Strict sample mode: {STRICT_SAMPLE_MODE}")
print(f"Sample size: {SAMPLE_SIZE} | split ratios: {SAMPLE_SPLIT_RATIOS} | seed: {SAMPLE_SEED}")
print(f"Processed root: {PROCESSED_DIR}")
print(f"Feature extraction batch size: {FEATURE_BATCH_SIZE}")
print(f"Task 1 config: batch_size={TASK1_BATCH_SIZE}, lr={TASK1_LR}, max_epochs={TASK1_MAX_EPOCHS}")
print(f"Task 2 config: batch_size={TASK2_BATCH_SIZE}, lr={TASK2_LR}, max_epochs={TASK2_MAX_EPOCHS}")


def require_files(paths, label):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Thiếu {label}: {missing}")


def ensure_nonempty_cache(cache_path):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    if cache_path.stat().st_size == 0:
        return False
    try:
        cache = torch.load(cache_path, map_location="cpu")
        return bool(cache)
    except Exception:
        return False


def get_feature_output(task_name, split_name):
    return (TASK1_FEATURE_DIR if task_name == "task1" else TASK2_FEATURE_DIR) / f"{split_name}_features.pt"

CONFIGURATION
project_name: visual-genome-caption
seed: 42
device: cuda
paths:
  data_dir: data
  raw_dir: data/raw
  processed_dir: data/processed
  feature_dir: data/features
  checkpoint_dir: checkpoints
  log_dir: logs
dataset:
  repo_id: ranjaykrishna/visual_genome
  cache_dir: data/raw/hf_cache
  image_dir: data/raw/images
image:
  roi_size: 224
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
split:
  train: 0.7
  val: 0.15
  test: 0.15
training:
  batch_size: 64
  num_workers: 4
  pin_memory: true
  max_epochs: 30
  early_stopping_patience: 5
  gradient_clip_val: 1.0
  log_every_n_steps: 50
optimizer:
  name: adamw
  lr: 0.0001
  weight_decay: 0.0001
scheduler:
  name: cosine
  warmup_epochs: 2
logging:
  use_tensorboard: true
  use_wandb: false
  wandb_project: vg-caption
pipeline:
  training_mode: both
  download_data: true
  pre_extract_features: true
sampling:
  strict_mode: true
  sample_size: 200
  split_ratios:
  - 0.8
  - 0.1
  - 0.1
  seed: 42

## 2. Load Data

In [19]:
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download

image_dir = PROJECT_ROOT / BASE_CONFIG.dataset.image_dir

if DOWNLOAD_DATA:
    print("Đang tải metadata Visual Genome...")
    download_and_extract_metadata(raw_dir=str(RAW_DIR), keep_zip=False)

    raw_status = verify_download(raw_dir=str(RAW_DIR))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(f"Thiếu metadata sau khi tải: {missing_metadata}")

    with open(RAW_DIR / "image_data.json", "r", encoding="utf-8") as f:
        img_data = json.load(f)

    if IMAGE_DOWNLOAD_MODE == 'sample':
        sample_ids = [img['image_id'] for img in img_data[:SAMPLE_SIZE]]
        print(f"Bắt đầu tải bộ sample {len(sample_ids)} ảnh...")
        downloaded_images = download_vg_images(sample_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError("Không tải được ảnh mẫu nào.")
    elif IMAGE_DOWNLOAD_MODE == 'all':
        all_ids = [img['image_id'] for img in img_data]
        print(f"Bắt đầu tải toàn bộ {len(all_ids)} ảnh...")
        downloaded_images = download_vg_images(all_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError("Không tải được ảnh nào.")
    elif IMAGE_DOWNLOAD_MODE == 'none':
        print("Bỏ qua tải ảnh theo cấu hình.")
    else:
        raise ValueError("IMAGE_DOWNLOAD_MODE phải là 'none', 'sample', hoặc 'all'.")
else:
    print("Bỏ qua tải mới, chỉ kiểm tra dữ liệu hiện có...")
    raw_status = verify_download(raw_dir=str(RAW_DIR))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(
            "Thiếu dữ liệu RAW cần thiết: " + ", ".join(missing_metadata) +
            ". Hãy bật DOWNLOAD_DATA = True hoặc đặt đúng thư mục data/raw."
        )

Đang tải metadata Visual Genome...
[Skip] objects.json đã tồn tại.
[Skip] attributes.json đã tồn tại.
[Skip] relationships.json đã tồn tại.
[Skip] image_data.json đã tồn tại.

--- Kiểm tra dữ liệu RAW ---
  ✅  objects.json (412.6 MB)
  ✅  attributes.json (462.6 MB)
  ✅  relationships.json (709.6 MB)
  ✅  image_data.json (17.6 MB)
Bỏ qua tải ảnh theo cấu hình.


In [20]:
if STRICT_SAMPLE_MODE:
    image_data_file = RAW_DIR / "image_data.json"
    require_files([image_data_file], "image_data.json")

    with open(image_data_file, "r", encoding="utf-8") as f:
        image_data = json.load(f)

    all_image_ids = [img["image_id"] for img in image_data]
    sample_count = min(SAMPLE_SIZE, len(all_image_ids))
    SAMPLE_IMAGE_IDS = random.Random(SAMPLE_SEED).sample(all_image_ids, sample_count)
    print(f"Đã chọn trước {len(SAMPLE_IMAGE_IDS)} image_id cho sample strict.")
else:
    SAMPLE_IMAGE_IDS = None
    print("Bỏ qua strict sample; sẽ dùng toàn bộ dữ liệu theo split mặc định.")

Đã chọn trước 200 image_id cho sample strict.


## 3. Data Preprocessing (Tạo Vocab & Split)

In [21]:
from src.data.preprocessing import build_vocab_and_splits

if TRAINING_MODE in ['task1', 'both']:
    build_vocab_and_splits(
        task='task1',
        raw_dir=str(RAW_DIR),
        processed_dir=str(PROCESSED_DIR),
        max_objects=int(TASK1_CONFIG.labels.max_objects),
        max_attributes=int(TASK1_CONFIG.labels.max_attributes),
        sample_image_ids=SAMPLE_IMAGE_IDS if STRICT_SAMPLE_MODE else None,
        split_by_image_id=STRICT_SAMPLE_MODE,
        split_ratios=PREPROCESS_SPLIT_RATIOS,
        seed=SAMPLE_SEED,
    )
    require_files(
        [
            TASK1_PROCESSED_DIR / 'object_vocab.json',
            TASK1_PROCESSED_DIR / 'attribute_vocab.json',
            TASK1_PROCESSED_DIR / 'train' / 'annotations.json',
            TASK1_PROCESSED_DIR / 'val' / 'annotations.json',
            TASK1_PROCESSED_DIR / 'test' / 'annotations.json',
        ],
        'Task 1 processed files'
    )

if TRAINING_MODE in ['task2', 'both']:
    build_vocab_and_splits(
        task='task2',
        raw_dir=str(RAW_DIR),
        processed_dir=str(PROCESSED_DIR),
        max_relations=int(TASK2_CONFIG.labels.max_relations),
        sample_image_ids=SAMPLE_IMAGE_IDS if STRICT_SAMPLE_MODE else None,
        split_by_image_id=STRICT_SAMPLE_MODE,
        split_ratios=PREPROCESS_SPLIT_RATIOS,
        seed=SAMPLE_SEED,
    )
    require_files(
        [
            TASK2_PROCESSED_DIR / 'relation_vocab.json',
            TASK2_PROCESSED_DIR / 'train' / 'annotations.json',
            TASK2_PROCESSED_DIR / 'val' / 'annotations.json',
            TASK2_PROCESSED_DIR / 'test' / 'annotations.json',
        ],
        'Task 2 processed files'
    )

--- [Task 1] Bắt đầu Preprocessing ---
Đang load objects file...
Đang load attributes file...
[Task 1] Giới hạn preprocessing trên 200 image_id đã chọn.
Đang đếm tần suất object & attributes...


Parsing: 100%|██████████| 108077/108077 [00:04<00:00, 25519.96it/s]


Đang mapping dữ liệu sang integer indices...
Tổng hợp được 5342 samples hợp lệ.
--- Hoàn tất Task 1 Preprocessing ---
--- [Task 2] Bắt đầu Preprocessing ---
[Task 2] Giới hạn preprocessing trên 200 image_id đã chọn.


Parsing Relationships: 100%|██████████| 108077/108077 [00:00<00:00, 1877470.71it/s]


Đang mapping dữ liệu sang integer indices...
--- Hoàn tất Task 2 Preprocessing ---


## 4. Trích Xuất Đặc Trưng (Feature Extraction)
Chỉ xuất đặc trưng từ những ảnh `.jpg` bạn đã tải về thư mục `images`.

In [22]:
from pathlib import Path
import json

from src.data.download import download_vg_images
from src.features.feature_extractor import extract_task1_features, extract_task2_features


def collect_split_image_ids(processed_dir, split_names):
    image_ids = set()
    for split_name in split_names:
        annotation_file = Path(processed_dir) / split_name / 'annotations.json'
        if not annotation_file.exists():
            raise FileNotFoundError(f"Thiếu annotation: {annotation_file}")

        with open(annotation_file, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        samples = raw if isinstance(raw, list) else raw.get('samples', [])
        image_ids.update(sample['image_id'] for sample in samples)

    return sorted(image_ids)


def ensure_images_for_splits(task_name, processed_dir, split_names):
    image_ids = collect_split_image_ids(processed_dir, split_names)
    missing_ids = [img_id for img_id in image_ids if not (image_dir / f"{img_id}.jpg").exists()]

    print(f"[{task_name}] Tổng ảnh tham chiếu: {len(image_ids)} | thiếu local: {len(missing_ids)}")
    if missing_ids:
        print(f"[{task_name}] Đang tải bổ sung ảnh còn thiếu...")
        downloaded = download_vg_images(missing_ids, image_dir=str(image_dir))
        if len(downloaded) < len(missing_ids):
            print(f"[Warning] {task_name}: còn {len(missing_ids) - len(downloaded)} ảnh chưa tải được.")


if PRE_EXTRACT_FEATURES:
    TASK1_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    TASK2_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    if TRAINING_MODE in ['task1', 'both']:
        ensure_images_for_splits('Task 1', TASK1_PROCESSED_DIR, ['train', 'val', 'test'])
        print('Extracting features Task 1...')
        for split_name in ['train', 'val', 'test']:
            extract_task1_features(
                annotation_file=str(TASK1_PROCESSED_DIR / split_name / 'annotations.json'),
                image_dir=str(image_dir),
                output_file=str(get_feature_output('task1', split_name)),
                backbone=str(TASK1_CONFIG.backbone.name),
                pretrained=bool(TASK1_CONFIG.backbone.pretrained),
                batch_size=FEATURE_BATCH_SIZE,
                device=DEVICE,
                resize_size=FEATURE_RESIZE_SIZE,
                crop_size=FEATURE_CROP_SIZE,
                mean=FEATURE_MEAN,
                std=FEATURE_STD,
            )
            if not ensure_nonempty_cache(get_feature_output('task1', split_name)):
                raise RuntimeError(f'Task 1 feature cache rỗng: {get_feature_output("task1", split_name)}')

    if TRAINING_MODE in ['task2', 'both']:
        ensure_images_for_splits('Task 2', TASK2_PROCESSED_DIR, ['train', 'val', 'test'])
        print('Extracting features Task 2...')
        for split_name in ['train', 'val', 'test']:
            extract_task2_features(
                annotation_file=str(TASK2_PROCESSED_DIR / split_name / 'annotations.json'),
                image_dir=str(image_dir),
                output_file=str(get_feature_output('task2', split_name)),
                backbone=str(TASK2_CONFIG.backbone.name),
                pretrained=bool(TASK2_CONFIG.backbone.pretrained),
                batch_size=FEATURE_BATCH_SIZE,
                device=DEVICE,
                resize_size=FEATURE_RESIZE_SIZE,
                crop_size=FEATURE_CROP_SIZE,
                mean=FEATURE_MEAN,
                std=FEATURE_STD,
            )
            if not ensure_nonempty_cache(get_feature_output('task2', split_name)):
                raise RuntimeError(f'Task 2 feature cache rỗng: {get_feature_output("task2", split_name)}')
else:
    print('Bỏ qua pre-extraction theo cấu hình PRE_EXTRACT_FEATURES = False.')

[Task 1] Tổng ảnh tham chiếu: 198 | thiếu local: 198
[Task 1] Đang tải bổ sung ảnh còn thiếu...
[Images] 0 ảnh đã có | 198 ảnh cần tải


[Images] Tải thành công: 198/198 ảnh mới
Extracting features Task 1...
Extracting Task 1 features using resnet50...


Processing images: 100%|██████████| 158/158 [02:18<00:00,  1.14it/s]


Saved 4494 features to c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\train_features.pt
Extracting Task 1 features using resnet50...


Processing images: 100%|██████████| 20/20 [00:14<00:00,  1.34it/s]


...

Saved 3223 features to c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task2\features\train_features.pt
Extracting Task 2 features using resnet50...


Processing images: 100%|██████████| 19/19 [00:08<00:00,  2.12it/s]


Saved 343 features to c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task2\features\val_features.pt
Extracting Task 2 features using resnet50...


Processing images: 100%|██████████| 20/20 [00:10<00:00,  1.98it/s]

Saved 387 features to c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task2\features\test_features.pt


## 5. Huấn Luyện (Training)

In [23]:
if TRAINING_MODE in ['task1', 'both']:
    print("========== BẮT ĐẦU TRAINING TASK 1 ==========")
    from src.training.task1_trainer import Task1Trainer
    from src.models.task1.object_classifier import ObjectClassifier
    from src.models.task1.attribute_classifier import AttributeClassifier
    from src.data.task1_dataset import build_task1_datasets
    from torch.utils.data import DataLoader
    
    print("Khởi tạo Dataset...")
    train_ds, val_ds, test_ds = build_task1_datasets(
        processed_dir=str(TASK1_PROCESSED_DIR),
        image_dir=str(image_dir),
        roi_size=TASK1_ROI_SIZE,
        use_feature_cache=PRE_EXTRACT_FEATURES,
        feature_cache_dir=str(TASK1_FEATURE_DIR),
        max_samples=MAX_SAMPLES,
        train_horizontal_flip_p=float(TASK1_CONFIG.augmentation.random_horizontal_flip),
        train_color_jitter=bool(TASK1_CONFIG.augmentation.color_jitter.enabled),
        train_brightness=float(TASK1_CONFIG.augmentation.color_jitter.brightness),
        train_contrast=float(TASK1_CONFIG.augmentation.color_jitter.contrast),
        train_saturation=float(TASK1_CONFIG.augmentation.color_jitter.saturation),
        train_hue=float(TASK1_CONFIG.augmentation.color_jitter.hue),
        train_random_erasing_p=float(TASK1_CONFIG.augmentation.random_erasing_p),
        train_resize_delta=int(TASK1_CONFIG.augmentation.resize_delta),
        mean=FEATURE_MEAN,
        std=FEATURE_STD,
    )
    
    obj_model = ObjectClassifier(
        num_classes=train_ds.num_objects,
        feature_dim=int(TASK1_CONFIG.backbone.feature_dim),
        hidden_dim=TASK1_OBJECT_HIDDEN_DIM,
        dropout=TASK1_OBJECT_DROPOUT,
        num_layers=TASK1_OBJECT_NUM_LAYERS,
        device=DEVICE,
    )
    attr_model = AttributeClassifier(
        num_attributes=train_ds.num_attributes,
        feature_dim=int(TASK1_CONFIG.backbone.feature_dim),
        hidden_dim=TASK1_ATTRIBUTE_HIDDEN_DIM,
        dropout=TASK1_ATTRIBUTE_DROPOUT,
        num_layers=TASK1_ATTRIBUTE_NUM_LAYERS,
        device=DEVICE,
    )
    
    obj_opt = torch.optim.AdamW(obj_model.parameters(), lr=TASK1_LR, weight_decay=float(BASE_CONFIG.optimizer.weight_decay))
    attr_opt = torch.optim.AdamW(attr_model.parameters(), lr=TASK1_LR, weight_decay=float(BASE_CONFIG.optimizer.weight_decay))
    
    train_loader = DataLoader(
        train_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=True,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    
    trainer1 = Task1Trainer(
        device=DEVICE,
        object_model=obj_model,
        attribute_model=attr_model,
        train_loader=train_loader,
        val_loader=val_loader,
        object_optimizer=obj_opt,
        attribute_optimizer=attr_opt,
        object_weight=TASK1_OBJECT_WEIGHT,
        attribute_weight=TASK1_ATTRIBUTE_WEIGHT,
        attribute_pos_weight=torch.full((train_ds.num_attributes,), TASK1_ATTRIBUTE_POS_WEIGHT, dtype=torch.float32),
        max_epochs=TASK1_MAX_EPOCHS,
        early_stopping_patience=int(BASE_CONFIG.training.early_stopping_patience),
        gradient_clip_val=float(BASE_CONFIG.training.gradient_clip_val),
        log_every_n_steps=int(BASE_CONFIG.training.log_every_n_steps),
        use_amp=(DEVICE == 'cuda'),
    )
    print("Bắt đầu Epochs...")
    task1_val_metrics = trainer1.train()
    print("Hoàn tất Training Task 1.")


if TRAINING_MODE in ['task2', 'both']:
    print("========== BẮT ĐẦU TRAINING TASK 2 ==========")
    from src.training.task2_trainer import Task2Trainer
    from src.models.task2.relation_classifier import RelationClassifier
    from src.data.task2_dataset import build_task2_datasets
    from torch.utils.data import DataLoader
    
    print("Khởi tạo Dataset...")
    train_ds2, val_ds2, test_ds2 = build_task2_datasets(
        processed_dir=str(TASK2_PROCESSED_DIR),
        image_dir=str(image_dir),
        roi_size=TASK2_ROI_SIZE,
        use_feature_cache=PRE_EXTRACT_FEATURES,
        feature_cache_dir=str(TASK2_FEATURE_DIR),
        use_spatial_features=bool(TASK2_CONFIG.spatial.use_spatial_features),
        max_samples=MAX_SAMPLES,
        train_horizontal_flip_p=float(TASK2_CONFIG.augmentation.random_horizontal_flip),
        train_color_jitter=bool(TASK2_CONFIG.augmentation.color_jitter.enabled),
        train_brightness=float(TASK2_CONFIG.augmentation.color_jitter.brightness),
        train_contrast=float(TASK2_CONFIG.augmentation.color_jitter.contrast),
        train_saturation=float(TASK2_CONFIG.augmentation.color_jitter.saturation),
        train_hue=float(TASK2_CONFIG.augmentation.color_jitter.hue),
        train_random_erasing_p=float(TASK2_CONFIG.augmentation.random_erasing_p),
        train_resize_delta=int(TASK2_CONFIG.augmentation.resize_delta),
        mean=FEATURE_MEAN,
        std=FEATURE_STD,
    )
    
    rel_model = RelationClassifier(
        num_relations=train_ds2.num_relations,
        feature_dim=int(TASK2_CONFIG.backbone.feature_dim),
        spatial_dim=int(TASK2_CONFIG.spatial.spatial_dim),
        hidden_dim=TASK2_HIDDEN_DIM,
        dropout=TASK2_DROPOUT,
        num_layers=TASK2_NUM_LAYERS,
        attention_heads=TASK2_ATTENTION_HEADS,
        fusion_method=TASK2_FUSION_METHOD,
        device=DEVICE,
    )
    rel_opt = torch.optim.AdamW(rel_model.parameters(), lr=TASK2_LR, weight_decay=float(BASE_CONFIG.optimizer.weight_decay))
    
    train_loader2 = DataLoader(
        train_ds2,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=True,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    val_loader2 = DataLoader(
        val_ds2,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    test_loader2 = DataLoader(
        test_ds2,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    
    trainer2 = Task2Trainer(
        device=DEVICE,
        model=rel_model,
        train_loader=train_loader2,
        val_loader=val_loader2,
        optimizer=rel_opt,
        label_smoothing=float(TASK2_CONFIG.loss.label_smoothing),
        max_epochs=TASK2_MAX_EPOCHS,
        early_stopping_patience=int(BASE_CONFIG.training.early_stopping_patience),
        gradient_clip_val=float(BASE_CONFIG.training.gradient_clip_val),
        log_every_n_steps=int(BASE_CONFIG.training.log_every_n_steps),
        use_amp=(DEVICE == 'cuda'),
    )
    print("Bắt đầu Epochs...")
    task2_val_metrics = trainer2.train()
    print("Hoàn tất Training Task 2.")

========== BẮT ĐẦU TRAINING TASK 1 ==========
Khởi tạo Dataset...
[Task1Dataset] Loaded 4494 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\train\annotations.json
[Task1Dataset] Loading feature cache: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\train_features.pt


2026-04-04 22:49:06 [INFO] TensorBoard logging to: logs\tensorboard\vg_caption
2026-04-04 22:49:06 [INFO] Starting training for 40 epochs
2026-04-04 22:49:06 [INFO] Model: ObjectClassifier
2026-04-04 22:49:06 [INFO] Train samples: 4494
2026-04-04 22:49:06 [INFO] Val samples: 444


[Task1Dataset] Loaded 4494 cached features
ObjectAttributeDataset [train]: 4494 ROIs | 158 images | 301 objects | 201 attributes | cache=✅
[Task1Dataset] Loaded 444 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\val\annotations.json
[Task1Dataset] Loading feature cache: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\val_features.pt
[Task1Dataset] Loaded 444 cached features
ObjectAttributeDataset [val]: 444 ROIs | 20 images | 301 objects | 201 attributes | cache=✅
[Task1Dataset] Loaded 404 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\test\annotations.json
[Task1Dataset] Loading feature cache: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\test_features.pt
[Task1Dataset] Loaded 404 cached features
ObjectAttributeDataset [test]: 404 ROIs | 20 images | 301 objects | 201 attributes | cache=✅
Bắt đầu Epochs...


2026-04-04 22:49:19 [INFO] Step 1: batch/batch_loss=6.4056, batch/batch_object_loss=5.6397, batch/batch_attribute_loss=0.6816, batch/batch_idx=0
2026-04-04 22:49:25 [INFO] Step 0: epoch/train_loss=5.5925, epoch/lr=0.0005, epoch/object_accuracy=0.0338, epoch/object_f1=0.0052, epoch/attribute_accuracy=0.5766, epoch/attribute_f1=0.0006, epoch/val_loss=5.4177


Removed old checkpoint: checkpoints\task1_object\task1_object_epoch_15.pth
Removed old checkpoint: checkpoints\task1_object\task1_object_epoch_12.pth
Checkpoint saved: checkpoints\task1_object\task1_object_epoch_0.pth
Removed old checkpoint: checkpoints\task1_attribute\task1_attribute_epoch_15.pth
Checkpoint saved: checkpoints\task1_attribute\task1_attribute_epoch_0.pth


...

Removed old checkpoint: checkpoints\task2\task2_epoch_5.pth
Checkpoint saved: checkpoints\task2\task2_epoch_10.pth


2026-04-04 23:00:10 [INFO] Step 562: batch/batch_loss=1.7205, batch/batch_idx=0
2026-04-04 23:00:11 [INFO] Step 612: batch/batch_loss=1.5589, batch/batch_idx=50
2026-04-04 23:00:16 [INFO] Step 11: epoch/train_loss=1.7146, epoch/lr=0.0003, epoch/accuracy=0.4023, epoch/f1=0.0482, epoch/val_loss=3.0986
2026-04-04 23:00:16 [INFO] Early stopping at epoch 11
2026-04-04 23:00:16 [INFO] Loaded best checkpoint


Removed old checkpoint: checkpoints\task2\task2_epoch_6.pth
Checkpoint saved: checkpoints\task2\task2_epoch_11.pth
Checkpoint loaded: checkpoints\task2\task2_epoch_9.pth
Epoch: 9, Loss: 1.8250962916542501


2026-04-04 23:00:21 [INFO] Training completed


Hoàn tất Training Task 2.


## 6. Sinh Caption & Đánh giá

In [24]:
from src.evaluation.metrics import compute_classification_metrics, compute_multilabel_metrics
from src.data.dataset import load_vocab
from src.models.caption.caption_generator import CaptionGenerator


def evaluate_task1_on_loader(object_model, attribute_model, loader):
    object_model.eval()
    attribute_model.eval()

    object_logits_list = []
    object_targets_list = []
    attribute_logits_list = []
    attribute_targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = batch['feature'] if 'feature' in batch else batch['image']

            object_logits = object_model(features)
            attribute_logits = attribute_model(features)

            object_logits_list.append(object_logits.cpu())
            object_targets_list.append(batch['object_label'].cpu())
            attribute_logits_list.append(attribute_logits.cpu())
            attribute_targets_list.append(batch['attribute_labels'].cpu())

    return {
        'object': compute_classification_metrics(torch.cat(object_logits_list), torch.cat(object_targets_list)),
        'attribute': compute_multilabel_metrics(torch.cat(attribute_logits_list), torch.cat(attribute_targets_list)),
    }


def evaluate_task2_on_loader(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = batch['feature'] if 'feature' in batch else batch['image']
            logits = model(features, batch['spatial'])
            logits_list.append(logits.cpu())
            targets_list.append(batch['relation_label'].cpu())

    return compute_classification_metrics(torch.cat(logits_list), torch.cat(targets_list))


if TRAINING_MODE in ['task1', 'both']:
    task1_test_metrics = evaluate_task1_on_loader(trainer1.object_model, trainer1.attribute_model, test_loader)
    print('Task 1 test metrics:')
    print(task1_test_metrics)

if TRAINING_MODE in ['task2', 'both']:
    task2_test_metrics = evaluate_task2_on_loader(trainer2.model, test_loader2)
    print('Task 2 test metrics:')
    print(task2_test_metrics)

if TRAINING_MODE == 'both':
    object_vocab = load_vocab(str(TASK1_PROCESSED_DIR / 'object_vocab.json'))
    attribute_vocab = load_vocab(str(TASK1_PROCESSED_DIR / 'attribute_vocab.json'))
    relation_vocab = load_vocab(str(TASK2_PROCESSED_DIR / 'relation_vocab.json'))

    captioner = CaptionGenerator(
        object_vocab=object_vocab,
        attribute_vocab=attribute_vocab,
        relation_vocab=relation_vocab,
    )

    demo_caption = captioner.generate_caption(
        subject_name='person',
        object_name='table',
        attributes=['small', 'wooden'],
        relation='standing near',
        template_idx=0,
    )
    print('Caption demo:')
    print(demo_caption)

Task 1 test metrics:
{'object': {'accuracy': 0.22772277227722773, 'f1': 0.09364530868745834}, 'attribute': {'accuracy': 0.41336633663366334, 'f1': 0.006963050440260421}}
Task 2 test metrics:
{'accuracy': 0.3798449612403101, 'f1': 0.04990315783749129}
Caption demo:
Person small, wooden standing near table.
